In [7]:
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine,String,text
import os
import psycopg


In [21]:
load_dotenv()

conn_path = os.getenv('CONN_PATH')

engine = create_engine(conn_path,pool_pre_ping = True)


In [22]:
with engine.connect() as conn:
    customer_orders_data = pd.read_sql_table(
        table_name='customer_orders', 
        con=conn, 
        schema='pizza_runner'
    )

# Display the DataFrame
customer_orders_data

,order_id,customer_id,pizza_id,exclusions,extras,order_time
0,1,101,1,,,2020-01-01 18:05:02
1,2,101,1,,,2020-01-01 19:00:52
2,3,102,1,,,2020-01-02 23:51:23
3,3,102,2,,None,2020-01-02 23:51:23
4,4,103,1,4,,2020-01-04 13:23:46
5,4,103,1,4,,2020-01-04 13:23:46
6,4,103,2,4,,2020-01-04 13:23:46
7,5,104,1,null,1,2020-01-08 21:00:29
8,6,101,2,null,null,2020-01-08 21:03:13
9,7,105,2,null,1,2020-01-08 21:20:29


In [24]:
import numpy as np

# 1. Standardize messy strings into true NaN values in-place
customer_orders_data.replace(['', 'null'], np.nan, inplace=True)

# 2. Fill those NaN values with a clean placeholder (like 'None' or an empty string)
customer_orders_data.fillna('None', inplace=True)

# View the cleaned data
customer_orders_data

,order_id,customer_id,pizza_id,exclusions,extras,order_time
0,1,101,1,None,None,2020-01-01 18:05:02
1,2,101,1,None,None,2020-01-01 19:00:52
2,3,102,1,None,None,2020-01-02 23:51:23
3,3,102,2,None,None,2020-01-02 23:51:23
4,4,103,1,4,None,2020-01-04 13:23:46
5,4,103,1,4,None,2020-01-04 13:23:46
6,4,103,2,4,None,2020-01-04 13:23:46
7,5,104,1,None,1,2020-01-08 21:00:29
8,6,101,2,None,None,2020-01-08 21:03:13
9,7,105,2,None,1,2020-01-08 21:20:29


In [25]:
# Insert your cleaned data back into the empty PostgreSQL table
with engine.connect() as conn:
    customer_orders_data.to_sql(
        name='customer_orders',
        con=conn,
        schema='pizza_runner',
        if_exists='append',  # Safe choice: appends data into the existing structure
        index=False          # Crucial: prevents creating an extra 'index' column
    )

print("Data successfully uploaded!")

Data successfully uploaded!


In [27]:
with engine.connect() as conn:
    runner_orders = pd.read_sql_table(
        table_name='runner_orders', 
        con=conn, 
        schema='pizza_runner'
    )

# Display the DataFrame
runner_orders

,order_id,runner_id,pickup_time,distance,duration,cancellation
0,1,1,2020-01-01 18:15:34,20km,32 minutes,
1,2,1,2020-01-01 19:10:54,20km,27 minutes,
2,3,1,2020-01-03 00:12:37,13.4km,20 mins,None
3,4,2,2020-01-04 13:53:03,23.4,40,None
4,5,3,2020-01-08 21:10:57,10,15,None
5,6,3,null,null,null,Restaurant Cancellation
6,7,2,2020-01-08 21:30:45,25km,25mins,null
7,8,2,2020-01-10 00:15:02,23.4 km,15 minute,null
8,9,2,null,null,null,Customer Cancellation
9,10,1,2020-01-11 18:50:20,10km,10minutes,null


In [ ]:
# 1. Standardize all variations of 'null', 'None', and blank spaces into true NaNs
runner_orders.replace(["null", "None", ""], np.nan, inplace=True)
runner_orders["cancellation"] = runner_orders["cancellation"].str.strip()

# 2. Clean 'distance' column: Strip 'km' and white spaces, then convert to float
runner_orders["distance"] = (
    runner_orders["distance"]
    .astype(str)
    .str.replace("km", "", case=False)
    .str.strip()
)
runner_orders["distance"] = pd.to_numeric(
    runner_orders["distance"], errors="coerce"
)

# 3. Clean 'duration' column: Strip 'minutes', 'minute', 'mins', 'min'
runner_orders["duration"] = (
    runner_orders["duration"]
    .astype(str)
    .str.replace("minutes", "", case=False)
    .str.replace("minute", "", case=False)
    .str.replace("mins", "", case=False)
    .str.replace("min", "", case=False)
    .str.strip()
)
runner_orders["duration"] = pd.to_numeric(
    runner_orders["duration"], errors="coerce"
)

# 4. Clean 'pickup_time' column: Convert to proper datetime format
runner_orders["pickup_time"] = pd.to_datetime(
    runner_orders["pickup_time"], errors="coerce"
)

# 5. Handle NaNs cleanly for SQL upload (keeps structure intact)
# For numeric/date columns, use None so SQL inserts true database NULL values
runner_orders = runner_orders.where(pd.notnull(runner_orders), None)



,order_id,runner_id,pickup_time,distance,duration,cancellation
0,1,1,2020-01-01 18:15:34,20.0,32.0,None
1,2,1,2020-01-01 19:10:54,20.0,27.0,None
2,3,1,2020-01-03 00:12:37,13.4,20.0,None
3,4,2,2020-01-04 13:53:03,23.4,40.0,None
4,5,3,2020-01-08 21:10:57,10.0,15.0,None
5,6,3,NaT,NaN,NaN,Restaurant Cancellation
6,7,2,2020-01-08 21:30:45,25.0,25.0,None
7,8,2,2020-01-10 00:15:02,23.4,15.0,None
8,9,2,NaT,NaN,NaN,Customer Cancellation
9,10,1,2020-01-11 18:50:20,10.0,10.0,None


In [32]:
# For the cancellation text column, fill remaining None with a clear indicator if desired
runner_orders['cancellation'] = runner_orders['cancellation'].fillna('No Cancellation')

# Display clean DataFrame
runner_orders

,order_id,runner_id,pickup_time,distance,duration,cancellation
0,1,1,2020-01-01 18:15:34,20.0,32.0,No Cancellation
1,2,1,2020-01-01 19:10:54,20.0,27.0,No Cancellation
2,3,1,2020-01-03 00:12:37,13.4,20.0,No Cancellation
3,4,2,2020-01-04 13:53:03,23.4,40.0,No Cancellation
4,5,3,2020-01-08 21:10:57,10.0,15.0,No Cancellation
5,6,3,NaT,NaN,NaN,Restaurant Cancellation
6,7,2,2020-01-08 21:30:45,25.0,25.0,No Cancellation
7,8,2,2020-01-10 00:15:02,23.4,15.0,No Cancellation
8,9,2,NaT,NaN,NaN,Customer Cancellation
9,10,1,2020-01-11 18:50:20,10.0,10.0,No Cancellation


In [34]:
# Insert your cleaned data back into the empty PostgreSQL table
with engine.connect() as conn:
    runner_orders.to_sql(
        name='runner_orders',
        con=conn,
        schema='pizza_runner',
        if_exists='append',  # Safe choice: appends data into the existing structure
        index=False          # Crucial: prevents creating an extra 'index' column
    )

print("Data successfully uploaded!")

Data successfully uploaded!
